In [ ]:
import pandas as pd

df = pd.read_csv('data/clean_fruki_data/dim_item_clean.csv')

df['categoria_texto'] = df['COD_DESC_FAMILIA'].astype(str).str.split('-').str[-1].str.strip().str.title()
df['nome_produto'] = df['categoria_texto'] + " " + df['ID_ITEM'].astype(str)
df_banco = df[['ID_ITEM', 'COD_DESC_FAMILIA', 'nome_produto']]
df_banco.to_csv('dim_item_banco.csv', index=False)

print(df_banco.head())

   ID_ITEM         COD_DESC_FAMILIA           nome_produto
0        1  0101 - REFRIGERANTE PET     Refrigerante Pet 1
1    19377   0114 - ENERGETICO LATA  Energetico Lata 19377
2    19382   0114 - ENERGETICO LATA  Energetico Lata 19382
3       13  0101 - REFRIGERANTE PET    Refrigerante Pet 13
4    22479             0104 - AGUAS            Aguas 22479


In [ ]:
df = pd.read_csv('data/clean_fruki_data/dim_cliente_clean.csv')

segmentos = {
    'Seg_630494605': 'Supermercado',
    'Seg_1069731367': 'Padaria',
    'Seg_1009330124': 'Restaurante',
    'Seg_1550002915': 'Posto de Combustível',
    'Seg_2137998953': 'Bar / Pub',
    'Seg_1190267637': 'Atacado',
    'Seg_2096903060': 'Distribuidora',
    'Seg_287235138': 'Lanchonete',
    'Seg_1943153311': 'Hotel / Pousada'
}

subsegmento = {
    'Supermercado': ['Hipermercado', 'Mercado de Bairro', 'Minimercado'],
    'Padaria': ['Padaria Tradicional', 'Cafeteria', 'Confeitaria'],
    'Restaurante': ['Pizzaria', 'Churrascaria', 'Buffet', 'Hamburgueria', 'Comida Japonesa'],
    'Posto de Combustível': ['Loja de Conveniência', 'Borracharia / Oficina'],
    'Bar / Pub': ['Pub', 'Bar de Bairro', 'Casa Noturna', 'Boteco'],
    'Atacado': ['Atacarejo', 'Atacadão Regional'],
    'Distribuidora': ['Distribuidora Exclusiva', 'Depósito de Bebidas'],
    'Lanchonete': ['Fast Food', 'Pastelaria', 'Trailer / Food Truck', 'Cantina Escolar'],
    'Hotel / Pousada': ['Hotel', 'Pousada', 'Motel', 'Hostel']
}

df['segmento_nome'] = df['segmento'].map(segmentos).fillna('Comércio Geral')

mapeamento_subseg = {}

agrupamento = df.groupby(['segmento_nome', 'SUBSEGMENTO']).size().reset_index()

for seg_nome in agrupamento['segmento_nome'].unique():
    hashes_subseg = agrupamento[agrupamento['segmento_nome'] == seg_nome]['SUBSEGMENTO'].tolist()
    
    lista_nomes = subsegmento.get(seg_nome, ['Geral', 'Padrão'])
    
    for i, hash_val in enumerate(hashes_subseg):
        nome_escolhido = lista_nomes[i % len(lista_nomes)]
        mapeamento_subseg[hash_val] = nome_escolhido

df['subsegmento_nome'] = df['SUBSEGMENTO'].map(mapeamento_subseg).fillna('Geral')

df.to_csv('dim_cliente.csv', index=False)

In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

df = pd.read_csv('dim_cliente.csv')

def corrigir_coordenada(valor):
    if pd.isna(valor):
        return None
    val_str = str(valor).replace(',', '.').strip()
    partes = val_str.split('.')
    if len(partes) > 1:
        return float(partes[0] + '.' + ''.join(partes[1:]))
    return float(val_str)

print("formatação...")
df['lat_'] = df['LATITUDE'].apply(corrigir_coordenada)
df['lon_'] = df['LONGITUDE'].apply(corrigir_coordenada)

geolocator = Nominatim(user_agent="fruki_b2b_motor_hackathon")
geocode = RateLimiter(geolocator.reverse, min_delay_seconds=1)

def buscar_dados_geograficos(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return 'Sem Cidade', 'Sem Estado', 'Sem CEP'
    
    try:
        local = geolocator.reverse(f"{lat}, {lon}", timeout=10)
        
        if local and 'address' in local.raw:
            addr = local.raw['address']
            
            cidade = addr.get('city', addr.get('town', addr.get('village', addr.get('municipality', 'Sem Cidade'))))
            estado = addr.get('state', 'Sem Estado')
            cep = addr.get('postcode', 'Sem CEP')
            
            return cidade, estado, cep
    except Exception as e:
        return 'Erro', 'Erro', 'Erro'
    
    return 'Sem Cidade', 'Sem Estado', 'Sem CEP'

print(f"buscando... {len(df)}")

cidades, estados, ceps = [], [], []

for index, row in df.iterrows():
    cid, est, cep = buscar_dados_geograficos(row['lat_corrigida'], row['lon_corrigida'])
    cidades.append(cid)
    estados.append(est)
    ceps.append(cep)
    
    if (index + 1) % 100 == 0:
        print(f"[{index + 1}/{len(df)}]")

df['cidade_real'] = cidades
df['estado_real'] = estados
df['cep_real'] = ceps

df_final = df.drop(columns=['LATITUDE', 'LONGITUDE', 'lat_corrigida', 'lon_corrigida'])
df_final.to_csv('dim_cliente_finalizada.csv', index=False)

print("\nAMEM!")

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('dim_cliente_finalizada.csv')

cidades_mock = [
    ('Lajeado', 'RS', '95900-000'),          
    ('Estrela', 'RS', '95880-000'),
    ('Porto Alegre', 'RS', '90000-000'),
    ('Canoas', 'RS', '92000-000'),           
    ('Novo Hamburgo', 'RS', '93300-000'),
    ('São Leopoldo', 'RS', '93000-000'),
    ('Gravataí', 'RS', '94000-000'),
    ('Viamão', 'RS', '94400-000'),
    ('Caxias do Sul', 'RS', '95000-000'),    
    ('Gramado', 'RS', '95670-000'),          
    ('Canela', 'RS', '95680-000'),
    ('Bento Gonçalves', 'RS', '95700-000'),
    ('Farroupilha', 'RS', '95180-000'),
    ('Capão da Canoa', 'RS', '95555-000'),   
    ('Torres', 'RS', '95560-000'),           
    ('Tramandaí', 'RS', '95590-000'),
    ('Imbé', 'RS', '95625-000'),
    ('Xangri-lá', 'RS', '95588-000'),
    ('Osório', 'RS', '95520-000'),
    ('Pelotas', 'RS', '96000-000'),          
    ('Rio Grande', 'RS', '96200-000'),       
    ('Bagé', 'RS', '96400-000'),
    ('Santa Maria', 'RS', '97000-000'),      
    ('Passo Fundo', 'RS', '99000-000'),      
    ('Erechim', 'RS', '99700-000'),
    ('Ijuí', 'RS', '98700-000'),
    ('Santa Cruz do Sul', 'RS', '96800-000'),
    ('Uruguaiana', 'RS', '97500-000'),       
    
    ('Florianópolis', 'SC', '88000-000'),    
    ('Balneário Camboriú', 'SC', '88330-000'), 
    ('Itajaí', 'SC', '88300-000'),
    ('Itapema', 'SC', '88220-000'),
    ('Navegantes', 'SC', '88370-000'),
    ('Garopaba', 'SC', '88495-000'),
    ('Imbituba', 'SC', '88780-000'),
    ('Laguna', 'SC', '88790-000'),
    ('Joinville', 'SC', '89200-000'),        
    ('Blumenau', 'SC', '89000-000'),         
    ('Brusque', 'SC', '88350-000'),
    ('Jaraguá do Sul', 'SC', '89250-000'),
    ('Rio do Sul', 'SC', '89160-000'),
    ('São José', 'SC', '88100-000'),
    ('Palhoça', 'SC', '88130-000'),
    ('Chapecó', 'SC', '89800-000'),          
    ('Concórdia', 'SC', '89700-000'),
    ('Criciúma', 'SC', '88800-000'),         
    ('Tubarão', 'SC', '88700-000'),
    ('Lages', 'SC', '88500-000')             
]

indices_aleatorios = np.random.choice(len(cidades_mock), size=len(df))

df['cidade_real'] = [cidades_mock[i][0] for i in indices_aleatorios]
df['estado_real'] = [cidades_mock[i][1] for i in indices_aleatorios]
df['cep_real'] = [cidades_mock[i][2] for i in indices_aleatorios]

df = df.drop(columns=['LATITUDE', 'LONGITUDE'], errors='ignore')

df.to_csv('dim_cliente_finalizada.csv', index=False)

In [ ]:
import pandas as pd

df = pd.read_csv('dim_cliente_finalizada.csv')

colunas_para_remover = ['ESTADO', 'CIDADE', 'bairro', 'segmento', 'SUBSEGMENTO', 'ENDERECO', 'CEP']
df = df.drop(columns=colunas_para_remover, errors='ignore')

df.to_csv('dim_cliente_banco.csv', index=False)

In [ ]:
import pandas as pd

df_item = pd.read_csv(r'data\data_format\itens-catalogo.csv') 
df_preco = pd.read_csv('data/clean_fruki_data/dim_tabela_preco_clean.csv')
ids_validos = df_item['ID_ITEM'].unique()
df_preco_limpa = df_preco[df_preco['ID_ITEM'].isin(ids_validos)].copy()
df_preco_limpa = df_preco_limpa.rename(columns={'TABELA_PRECO': 'id_tabela_preco'})
df_preco_limpa = df_preco_limpa.drop_duplicates(subset=['id_tabela_preco', 'ID_ITEM'], keep='last')

df_preco_limpa.to_csv('dim_tabela_preco_banco.csv', index=False)

In [ ]:
import pandas as pd
import numpy as np

df_cliente = pd.read_csv('dim_cliente.csv')
df_preco = pd.read_csv('dim_tabela_preco.csv')

tabelas_reais = df_preco['id_tabela_preco'].unique()
np.random.seed(42)
np.random.shuffle(tabelas_reais)

zonas = {
    'Litoral': ['Capão da Canoa', 'Torres', 'Tramandaí', 'Imbé', 'Xangri-lá', 'Osório', 'Florianópolis', 'Balneário Camboriú', 'Itajaí', 'Itapema', 'Navegantes', 'Garopaba', 'Imbituba', 'Laguna', 'Pelotas', 'Rio Grande'],
    'Serra': ['Caxias do Sul', 'Gramado', 'Canela', 'Bento Gonçalves', 'Farroupilha', 'Lages'],
    'Metropolitana': ['Porto Alegre', 'Canoas', 'Novo Hamburgo', 'São Leopoldo', 'Gravataí', 'Viamão', 'São José', 'Palhoça', 'Joinville'],
    'Interior': ['Lajeado', 'Estrela', 'Santa Maria', 'Passo Fundo', 'Erechim', 'Ijuí', 'Santa Cruz do Sul', 'Uruguaiana', 'Bagé', 'Chapecó', 'Concórdia', 'Criciúma', 'Tubarão', 'Blumenau', 'Brusque', 'Jaraguá do Sul', 'Rio do Sul']
}

tabs_divididas = np.array_split(tabelas_reais, 4)
tabs_por_zona = {
    'Litoral': tabs_divididas[0],
    'Serra': tabs_divididas[1],
    'Metropolitana': tabs_divididas[2],
    'Interior': tabs_divididas[3]
}

cidade_para_zona = {}
for zona, cidades in zonas.items():
    for cidade in cidades:
        cidade_para_zona[cidade] = zona

def atribuir_tabela(cidade):
    zona_da_cidade = cidade_para_zona.get(cidade, 'Interior')
    return np.random.choice(tabs_por_zona[zona_da_cidade])

df_cliente['ID_TABELA_PRECO'] = df_cliente['cidade_real'].apply(atribuir_tabela)
df_cliente.to_csv('dim_cliente_banco.csv', index=False)

C:\Users\w_sta\AppData\Local\Temp\ipykernel_26104\1389045630.py:11: UserWarning: you are shuffling a 'StringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  np.random.shuffle(tabelas_reais)


In [ ]:
import pandas as pd
import random

df_cliente = pd.read_csv('cliente.csv')
df_hierarquia = pd.read_csv('dim_hierarquia_comercial_clean.csv')

nomes = ['Ana', 'Carlos', 'João', 'Maria', 'Pedro', 'Lucas', 'Mariana', 'Juliana', 'Marcos', 'Fernanda', 'Rafael', 'Amanda', 'Marcelo', 'Camila', 'Rodrigo', 'Letícia', 'Felipe', 'Bruna', 'Diego', 'Patrícia', 'Thiago', 'Natália', 'Alexandre', 'Tatiane', 'Eduardo', 'Aline', 'Ricardo', 'Carolina', 'Leandro', 'Beatriz']
sobrenomes = ['Silva', 'Oliveira', 'Souza', 'Costa', 'Santos', 'Almeida', 'Rodrigues', 'Alves', 'Pereira', 'Lima', 'Gomes', 'Ribeiro', 'Martins', 'Carvalho', 'Mendes', 'Nunes', 'Barbosa', 'Pinto', 'Dias', 'Castro']

random.seed(42)
lista_nomes = [f"{n} {s}" for n in nomes for s in sobrenomes]
random.shuffle(lista_nomes)

reps_unicos = df_hierarquia['REPRESENTANTE'].unique()
mapa_nomes = {rep: lista_nomes[i] for i, rep in enumerate(reps_unicos)}

df_hierarquia['nome_vendedor'] = df_hierarquia['REPRESENTANTE'].map(mapa_nomes)
df_hierarquia = df_hierarquia.rename(columns={'ID_HIERARQUIA': 'id_hierarquia'})
ids_clientes = set(df_cliente['id_hierarquia'].dropna().unique())
ids_conhecidos_faltantes = set(range(9, 135))
todos_ids_necessarios = ids_clientes.union(ids_conhecidos_faltantes)
ids_hierarquia = set(df_hierarquia['id_hierarquia'].unique())
ids_orfaos = list(todos_ids_necessarios - ids_hierarquia)

novas_linhas = []
for missing_id in ids_orfaos:
    novas_linhas.append({
        'id_hierarquia': missing_id,
        'COD_REPRESENTANTE': 99999,
        'REPRESENTANTE': 'Rep_App_B2B',
        'nome_vendedor': 'Canal Digital B2B'
    })

if novas_linhas:
    df_novas_linhas = pd.DataFrame(novas_linhas)
    df_hierarquia = pd.concat([df_hierarquia, df_novas_linhas], ignore_index=True)

df_hierarquia.to_csv('dim_hierarquia_banco.csv', index=False)

Hierarquia pronta! Colunas originais mantidas. 161 IDs de App gerados.
